# Text-To-Video-AI: Colab quick start

Этот ноутбук рассчитан на сценарий **Runtime -> Run all**. По умолчанию он использует стабильный режим: локальные placeholder-картинки из `input_images` + бесплатный Silero TTS. ComfyUI и Z-Image Turbo GGUF включаются только если выбрать `VISUAL_BACKEND = "comfyui"`.

## 1. Clone repo / enter project

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/dstryukov/Text-To-Video-AI.git"
PROJECT_DIR = Path("/content/Text-To-Video-AI")

if Path("/content").exists():
    os.chdir("/content")
    if not PROJECT_DIR.exists():
        !git clone {REPO_URL}
    %cd /content/Text-To-Video-AI
else:
    PROJECT_DIR = Path.cwd()
    print(f"Not running in Colab. Using current project: {PROJECT_DIR}")

print(f"Project directory: {Path.cwd()}")

## 2. Install base dependencies

Базовая установка не включает тяжёлые F5/Fish/CosyVoice пакеты и не требует ComfyUI.

In [ ]:
!python -m pip install -U pip setuptools wheel
!python -m pip install -r requirements.txt
!python -m pip install -U huggingface_hub

## 3. Fix dependency compatibility if needed

In [ ]:
import sys
print(f"Python: {sys.version}")
print("audioop-lts is installed only on Python >= 3.13 via requirements marker.")

## 4. Hugging Face login

Токен не печатается, не пишется в `config.yaml` и не сохраняется в git. Пустой токен допустим для режимов без закрытых моделей.

In [ ]:
from getpass import getpass
import os
from huggingface_hub import login

hf_token = getpass("Введите Hugging Face token (можно оставить пустым): ").strip()

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
    login(token=hf_token)
    print("Hugging Face login successful.")
else:
    print("HF token не введён. Некоторые модели могут быть недоступны.")

## 5. Optional API keys

Если не указать LLM ключи, ноутбук создаст короткий demo-сценарий локально, чтобы render pipeline всё равно проверялся из коробки.

In [ ]:
from getpass import getpass
import os

def ask_secret(env_name, label):
    value = getpass(f"{label} (можно оставить пустым): ").strip()
    if value:
        os.environ[env_name] = value
        print(f"{env_name} установлен.")
    else:
        print(f"{env_name} не задан.")

ask_secret("GEMINI_API_KEY", "Gemini API key")
ask_secret("OPENAI_API_KEY", "OpenAI API key")
ask_secret("GROQ_API_KEY", "Groq API key")
ask_secret("PEXELS_API_KEY", "Pexels API key")

if os.getenv("GEMINI_API_KEY"):
    os.environ["LLM_PROVIDER"] = "gemini"
elif os.getenv("OPENAI_API_KEY"):
    os.environ["LLM_PROVIDER"] = "openai"
elif os.getenv("GROQ_API_KEY"):
    os.environ["LLM_PROVIDER"] = "groq"

USE_SAMPLE_SCENES = not any(os.getenv(name) for name in ["GEMINI_API_KEY", "OPENAI_API_KEY", "GROQ_API_KEY"])
print("USE_SAMPLE_SCENES =", USE_SAMPLE_SCENES)

## 6. Select mode

Colab default максимально стабильный: `image_folder` + `silero`.

In [ ]:
PROJECT_TOPIC = "Интересные факты о космосе"

VISUAL_BACKEND = "image_folder"
# варианты:
# "none"          — только аудио + субтитры на чёрном фоне
# "image_folder"  — брать картинки/видео из папки input_images
# "stock_video"   — стоковые видео через Pexels
# "comfyui"       — генерация через ComfyUI

TTS_BACKEND = "silero"
# варианты:
# "silero"        — стабильный бесплатный fallback
# "f5_tts"        — качественная озвучка, optional
# "local_tts_api" — внешний локальный API
# "audio_file"    — готовая озвучка
# "none"          — тишина

TTS_MODE = "per_scene"
# "per_scene" или "full_script"

TTS_QUALITY = "balanced"
# "draft" | "balanced" | "high"

MODEL_PRESET = "z_image_turbo_gguf"
# используется только если VISUAL_BACKEND="comfyui"

IMAGE_WIDTH = 576
IMAGE_HEIGHT = 1024
STEPS = 6
GUIDANCE = 0.0

print({
    "topic": PROJECT_TOPIC,
    "visual_backend": VISUAL_BACKEND,
    "tts_backend": TTS_BACKEND,
    "tts_mode": TTS_MODE,
    "model_preset": MODEL_PRESET,
})

## 7. Create demo input_images if empty

Если пользователь загрузил свои картинки или видео, они не перезаписываются.

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

input_dir = Path("input_images")
input_dir.mkdir(exist_ok=True)
media_exts = {".png", ".jpg", ".jpeg", ".webp", ".mp4", ".mov", ".avi", ".mkv"}
existing_media = [p for p in input_dir.iterdir() if p.is_file() and p.suffix.lower() in media_exts]

def create_placeholder(path, title, subtitle, color_a, color_b):
    width, height = 1080, 1920
    image = Image.new("RGB", (width, height), color_a)
    pixels = image.load()
    for y in range(height):
        ratio = y / max(1, height - 1)
        row = tuple(int(color_a[i] * (1 - ratio) + color_b[i] * ratio) for i in range(3))
        for x in range(width):
            pixels[x, y] = row

    draw = ImageDraw.Draw(image)
    try:
        title_font = ImageFont.truetype("DejaVuSans-Bold.ttf", 110)
        body_font = ImageFont.truetype("DejaVuSans.ttf", 54)
    except Exception:
        title_font = ImageFont.load_default()
        body_font = ImageFont.load_default()

    title_box = draw.textbbox((0, 0), title, font=title_font)
    subtitle_box = draw.textbbox((0, 0), subtitle, font=body_font)
    title_x = (width - (title_box[2] - title_box[0])) // 2
    subtitle_x = (width - (subtitle_box[2] - subtitle_box[0])) // 2
    draw.text((title_x, 780), title, fill="white", font=title_font)
    draw.text((subtitle_x, 930), subtitle, fill=(235, 245, 255), font=body_font)
    image.save(path, quality=95)

if existing_media:
    print(f"input_images уже содержит {len(existing_media)} медиафайл(ов). Ничего не перезаписываю.")
else:
    placeholders = [
        ("scene_1.png", "Scene 1", "Demo visual", (21, 31, 58), (14, 116, 144)),
        ("scene_2.png", "Scene 2", "Replace with your media", (68, 38, 87), (224, 107, 84)),
        ("scene_3.png", "Scene 3", "Render pipeline check", (20, 83, 45), (245, 179, 66)),
    ]
    for filename, title, subtitle, color_a, color_b in placeholders:
        create_placeholder(input_dir / filename, title, subtitle, color_a, color_b)
    print("Созданы demo placeholder-картинки в input_images/.")

## 8. Optional: install ComfyUI

In [ ]:
import subprocess
from pathlib import Path

def run_cmd(args, cwd=None, check=True):
    print("$", " ".join(map(str, args)))
    return subprocess.run(args, cwd=cwd, check=check)

if VISUAL_BACKEND == "comfyui":
    comfy_dir = Path("/content/ComfyUI")
    if not comfy_dir.exists():
        run_cmd(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", str(comfy_dir)])
    run_cmd(["python", "-m", "pip", "install", "-r", "requirements.txt"], cwd=comfy_dir)
else:
    print("VISUAL_BACKEND != 'comfyui': пропускаю установку ComfyUI.")

## 9. Optional: install GGUF custom node

После установки custom node сервер ComfyUI нужно запускать заново.

In [ ]:
from pathlib import Path

if VISUAL_BACKEND == "comfyui":
    comfy_dir = Path("/content/ComfyUI")
    nodes_dir = comfy_dir / "custom_nodes"
    gguf_dir = nodes_dir / "ComfyUI-GGUF"
    nodes_dir.mkdir(parents=True, exist_ok=True)
    if not gguf_dir.exists():
        run_cmd(["git", "clone", "https://github.com/city96/ComfyUI-GGUF.git", str(gguf_dir)])
    run_cmd(["python", "-m", "pip", "install", "-r", str(gguf_dir / "requirements.txt")], cwd=comfy_dir)
else:
    print("VISUAL_BACKEND != 'comfyui': пропускаю ComfyUI-GGUF.")

## 10. Optional: download Z-Image Turbo GGUF

Flux Schnell FP8 не скачивается по умолчанию. Для VAE `ae.safetensors` может понадобиться принять условия на странице black-forest-labs/FLUX.1-schnell.

In [ ]:
from pathlib import Path

if VISUAL_BACKEND == "comfyui":
    comfy_dir = Path("/content/ComfyUI")
    targets = {
        "unet/z-image-turbo-Q4_K_M.gguf": ["hf", "download", "unsloth/Z-Image-Turbo-GGUF", "z-image-turbo-Q4_K_M.gguf", "--local-dir", str(comfy_dir / "models" / "unet")],
        "clip/clip_l.safetensors": ["hf", "download", "comfyanonymous/flux_text_encoders", "clip_l.safetensors", "--local-dir", str(comfy_dir / "models" / "clip")],
        "clip/t5xxl_fp8_e4m3fn.safetensors": ["hf", "download", "comfyanonymous/flux_text_encoders", "t5xxl_fp8_e4m3fn.safetensors", "--local-dir", str(comfy_dir / "models" / "clip")],
        "vae/ae.safetensors": ["hf", "download", "black-forest-labs/FLUX.1-schnell", "ae.safetensors", "--local-dir", str(comfy_dir / "models" / "vae")],
    }
    for rel_path, cmd in targets.items():
        out_path = comfy_dir / "models" / rel_path
        out_path.parent.mkdir(parents=True, exist_ok=True)
        if out_path.exists():
            print(f"Already downloaded: {out_path}")
            continue
        result = subprocess.run(cmd, cwd="/content", text=True)
        if result.returncode != 0 and rel_path == "vae/ae.safetensors":
            print("Не удалось скачать ae.safetensors. Примите условия на странице https://huggingface.co/black-forest-labs/FLUX.1-schnell и повторите ячейку.")
else:
    print("VISUAL_BACKEND != 'comfyui': модели Z-Image не скачиваются.")

## 11. Optional: start ComfyUI

In [ ]:
import subprocess, time, requests
from pathlib import Path

if VISUAL_BACKEND == "comfyui":
    comfy_dir = Path("/content/ComfyUI")
    log_path = Path("/content/comfyui.log")
    log_file = open(log_path, "w", encoding="utf-8")
    process = subprocess.Popen(
        ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
        cwd=comfy_dir,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )

    for i in range(60):
        try:
            response = requests.get("http://127.0.0.1:8188/system_stats", timeout=2)
            if response.status_code == 200:
                print("ComfyUI is running.")
                break
        except Exception:
            pass
        time.sleep(2)
    else:
        print("ComfyUI did not start. Check /content/comfyui.log")
        !tail -100 /content/comfyui.log
else:
    print("VISUAL_BACKEND != 'comfyui': ComfyUI не запускается.")

## 12. Check ComfyUI files before Z-Image run

In [ ]:
from pathlib import Path

if VISUAL_BACKEND == "comfyui":
    required_files = [
        Path("/content/ComfyUI/models/unet/z-image-turbo-Q4_K_M.gguf"),
        Path("/content/ComfyUI/models/clip/clip_l.safetensors"),
        Path("/content/ComfyUI/models/clip/t5xxl_fp8_e4m3fn.safetensors"),
        Path("/content/ComfyUI/models/vae/ae.safetensors"),
        Path("workflows/z_image_turbo_gguf_6step.json"),
    ]
    missing = [str(path) for path in required_files if not path.exists()]
    if missing:
        print("Не хватает файлов для ComfyUI Z-Image режима:")
        for item in missing:
            print("-", item)
        print("app.py всё равно будет запущен с fallback_backend=image_folder, если основной comfyui backend не сработает.")
    else:
        print("Все файлы для Z-Image Turbo GGUF найдены.")
else:
    print("Проверка ComfyUI-файлов не нужна для текущего режима.")

## 13. Run app.py

In [ ]:
import json
import os
import subprocess
from pathlib import Path

def write_sample_scenes(topic):
    sample_path = Path("colab_sample_scenes.json")
    scenes = [
        {
            "scene_id": 1,
            "text": f"{topic}. Это короткая проверка пайплайна: озвучка, субтитры и визуальный фон.",
            "duration": 4.0,
            "prompt": "vertical cinematic star field, deep space, bright nebula",
            "negative_prompt": "blurry, low quality, watermark",
            "camera_motion": "slow_zoom_in",
            "subtitle": f"{topic}. Проверка пайплайна.",
            "visual_type": "image_prompts",
            "style_preset": "cinematic_realistic",
            "stock_query": "deep space nebula"
        },
        {
            "scene_id": 2,
            "text": "Если вы видите это видео, базовый Colab render работает без тяжёлых моделей.",
            "duration": 4.0,
            "prompt": "vertical futuristic observatory under stars, cinematic lighting",
            "negative_prompt": "blurry, low quality, watermark",
            "camera_motion": "slow_zoom_out",
            "subtitle": "Базовый Colab render работает.",
            "visual_type": "image_prompts",
            "style_preset": "cinematic_realistic",
            "stock_query": "starry observatory"
        }
    ]
    sample_path.write_text(json.dumps(scenes, ensure_ascii=False, indent=2), encoding="utf-8")
    return sample_path

cmd = [
    "python", "app.py", PROJECT_TOPIC,
    "--tts-backend", TTS_BACKEND,
    "--tts-mode", TTS_MODE,
    "--tts-quality", TTS_QUALITY,
    "--backend", VISUAL_BACKEND,
]

if USE_SAMPLE_SCENES:
    sample_scenes = write_sample_scenes(PROJECT_TOPIC)
    cmd = [
        "python", "app.py", str(sample_scenes),
        "--tts-backend", TTS_BACKEND,
        "--tts-mode", TTS_MODE,
        "--tts-quality", TTS_QUALITY,
        "--backend", VISUAL_BACKEND,
    ]

if VISUAL_BACKEND == "stock_video":
    cmd += ["--fallback-backend", "image_folder"]
elif VISUAL_BACKEND == "comfyui":
    cmd += [
        "--model-preset", MODEL_PRESET,
        "--acceleration", "turbo",
        "--image-width", str(IMAGE_WIDTH),
        "--image-height", str(IMAGE_HEIGHT),
        "--steps", str(STEPS),
        "--guidance", str(GUIDANCE),
        "--fallback-backend", "image_folder",
    ]
elif VISUAL_BACKEND == "image_folder":
    cmd += ["--fallback-backend", "none"]

print("$", " ".join(cmd))
subprocess.run(cmd, check=True)

## 14. Show/download rendered_video.mp4

In [ ]:
from pathlib import Path

root_video = Path("rendered_video.mp4")
if root_video.exists():
    print(f"Готово: {root_video.resolve()} ({root_video.stat().st_size / 1024 / 1024:.2f} MB)")
    try:
        from IPython.display import Video, display
        display(Video(str(root_video), embed=True, width=360))
    except Exception as exc:
        print(f"Preview недоступен: {exc}")
    try:
        from google.colab import files
        files.download(str(root_video))
    except Exception:
        print("Не Colab или download недоступен. Файл уже лежит в корне проекта.")
else:
    print("rendered_video.mp4 не найден. Проверьте лог предыдущей ячейки.")

## Command examples

Quick test, no visuals:

```bash
python app.py "$PROJECT_TOPIC" --tts-backend silero --tts-mode per_scene --backend none
```

Image folder default:

```bash
python app.py "$PROJECT_TOPIC" --tts-backend silero --tts-mode per_scene --backend image_folder
```

Stock video:

```bash
python app.py "$PROJECT_TOPIC" --tts-backend silero --tts-mode per_scene --backend stock_video --fallback-backend image_folder
```

ComfyUI Z-Image:

```bash
python app.py "$PROJECT_TOPIC" --tts-backend silero --tts-mode per_scene --backend comfyui --model-preset z_image_turbo_gguf --acceleration turbo --image-width 576 --image-height 1024 --steps 6 --guidance 0.0 --fallback-backend image_folder
```

F5 optional:

```bash
python app.py "$PROJECT_TOPIC" --tts-backend f5_tts --tts-mode full_script --tts-quality draft --reference-audio voices/reference.wav --backend image_folder
```